In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "kmader/skin-cancer-mnist-ham10000", 
          path="HAM10000_metadata.csv"
    )

In [ ]:

meta = pd.read_csv(path)
meta.head()

In [ ]:
meta.describe(include='all')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

In [ ]:
order = meta.dx.value_counts().index
plt.figure(figsize=(8, 4))
sns.countplot(data=meta, x='dx', order=order, hue='dx', palette='viridis', legend=False)
plt.title('Distribution of lesion types')
plt.xlabel('Diagnosis'); plt.ylabel('Number of images')
plt.tight_layout(); plt.show()

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - (k-1)*(r-1)/(n-1))
    rcorr = r - (r-1)**2/(n-1)
    kcorr = k - (k-1)**2/(n-1)
    return np.sqrt(phi2corr / max(1e-10, min(kcorr-1, rcorr-1)))

cats = ['dx', 'dx_type', 'sex', 'localization']
V = pd.DataFrame(
    [[cramers_v(meta[a], meta[b]) for b in cats] for a in cats],
    index=cats, columns=cats
)

plt.figure(figsize=(6, 5))
sns.heatmap(V, annot=True, fmt='.2f', cmap='rocket_r', vmin=0, vmax=1, square=True)
plt.title("Cramér's V association between categorical variables")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=meta, x='dx', y='age', order=order, hue='dx', palette='viridis', legend=False)
plt.title('Age distribution by lesion type')
plt.xlabel('Diagnosis'); plt.ylabel('Age (years)')
plt.tight_layout(); plt.show()

In [ ]:
ct = pd.crosstab(meta.localization, meta.dx, normalize='index')
plt.figure(figsize=(8, 6))
sns.heatmap(ct, annot=True, fmt='.2f', cmap='mako')
plt.title('Lesion type distribution by body site (row-normalised)')
plt.tight_layout(); plt.show()

In [ ]:
counts = meta.groupby('lesion_id').size().value_counts().sort_index()
plt.figure(figsize=(6, 3.5))
counts.plot(kind='bar', color='steelblue')
plt.title('Images per unique lesion')
plt.xlabel('Number of images'); plt.ylabel('Number of lesions')
plt.tight_layout(); plt.show()
print(f"{meta.lesion_id.nunique()} unique lesions across {len(meta)} images")